# Libraries

In [1]:
import asyncio
import pandas as pd
import re
from collections import Counter
from datetime import datetime
from twikit import Client

In [2]:
# ====================== LOGIN CREDENTIALS ======================
USERNAME = "Uncle_Tolu_"
EMAIL = "Towllumuzik@gmail.com"
PASSWORD = "Danayobami08!"

In [3]:
# ====================== PARAMETERS ======================
handle = "CommBank"
max_tweets = 100

In [4]:



# ====================== INITIALIZE CLIENT ======================
client = Client("en-US")   # Language code

async def login():
    """Login to X using credentials"""
    print("Logging into X...")
    await client.login(
        auth_info_1=USERNAME,
        auth_info_2=EMAIL,
        password=PASSWORD
    )
    print("Login successful!")

async def fetch_tweets_from_user(max_tweets=100, tweet_type="Tweets"):
    """
    Fetch tweets from a user using Twikit.
    tweet_type options: 'Tweets', 'Replies', 'Media', 'Likes'
    """
    print(f"Fetching up to {max_tweets} {tweet_type.lower()} from @{handle}...")

    # First, get the user object by screen name
    user = await client.get_user_by_screen_name(handle)

    all_tweets = []
    collected = 0
    cursor = None

    while collected < max_tweets:
        # Get tweets (Twikit returns a Result object that supports .next())
        result = await client.get_user_tweets(
            user_id=user.id,
            tweet_type=tweet_type,
            count=min(40, max_tweets - collected),   # Twikit recommends ~20-40 per call
            cursor=cursor
        )

        if not result or len(result) == 0:
            print("No more tweets available.")
            break

        for tweet in result:
            row = {
                "tweet_id": tweet.id,
                "text": tweet.text,
                "created_at": tweet.created_at,          # string like "Mon Apr 06 12:34:56 +0000 2026"
                "lang": getattr(tweet, "lang", None),
                "likes": getattr(tweet, "favorite_count", 0),
                "retweets": getattr(tweet, "retweet_count", 0),
                "replies": getattr(tweet, "reply_count", 0),
                "quotes": getattr(tweet, "quote_count", 0),
                "views": getattr(tweet, "view_count", None),   # may not always be available
                "author_id": tweet.user.id,
                "author_name": tweet.user.name,
                "author_username": tweet.user.screen_name,
                "author_followers": getattr(tweet.user, "followers_count", None),
                "author_following": getattr(tweet.user, "friends_count", None),
                "author_verified": getattr(tweet.user, "verified", False),
                "is_reply": tweet.in_reply_to_status_id is not None,
                "conversation_id": getattr(tweet, "conversation_id", None),
                "source": getattr(tweet, "source", None),
            }
            all_tweets.append(row)
            collected += 1

            if collected >= max_tweets:
                break

        # Get next cursor for pagination
        cursor = result.next_cursor if hasattr(result, "next_cursor") else None

        if not cursor:
            break

        await asyncio.sleep(1)  # Be respectful to avoid rate limits / blocks

    print(f"Successfully collected {len(all_tweets)} tweets from @{handle}")
    return all_tweets







In [5]:
# ====================== MAIN FUNCTION ======================
async def main():
    await login()

    tweets_list = await fetch_tweets_from_user(
        max_tweets=100,
        tweet_type="Tweets"          # Change to "Replies" if you want replies too
    )

    if not tweets_list:
        print("No tweets were collected.")
        return

    # Convert to DataFrame
    df = pd.DataFrame(tweets_list)

    # Basic processing
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    df["date"] = df["created_at"].dt.date
    df["hour"] = df["created_at"].dt.hour

    # Save to CSV
    filename = f"commbank_tweets_twikit_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    df.to_csv(filename, index=False, encoding="utf-8")
    print(f"Data saved to → {filename}")

    # Quick summary
    print("\n=== Quick Summary ===")
    print(f"Total tweets collected : {len(df)}")
    print(f"Date range             : {df['created_at'].min()} → {df['created_at'].max()}")
    print(f"Average likes          : {df['likes'].mean():.1f}")
    print(f"Most active hour       : {df['hour'].mode()[0]}:00")

    # Simple word frequency (basic cleaning)
    all_text = " ".join(df["text"].str.lower())
    words = re.findall(r'\b\w+\b', all_text)
    word_freq = Counter(words)
    print("\nTop 10 most frequent words:")
    print(word_freq.most_common(10))

    print("\nSample of collected tweets:")
    print(df[["created_at", "text", "likes", "retweets"]].head())

In [6]:
# ====================== RUN THE SCRIPT ======================
await main()

Logging into X...


Exception: Couldn't get KEY_BYTE indices

In [ ]:
async def fetch_tweets():

    client = Client(language='en-US')

    # Login to X
    await client.login(
        auth_info_1=USERNAME,
        auth_info_2=EMAIL,
        password=PASSWORD
    )

    user = await client.get_user_by_screen_name(handle)

    tweets = await user.get_tweets('Tweets')

    tweets_list = []
    count = 0

    for tweet in tweets:
        if count >= max_tweets:
            break

        tweets_list.append({
            "tweet_id": tweet.id,
            "text": tweet.text,
            "created_at": tweet.created_at,
            "likes": tweet.favorite_count,
            "retweets": tweet.retweet_count,
            "replies": tweet.reply_count,
            "quotes": tweet.quote_count,
            "lang": tweet.lang,
            "source": tweet.source
        })

        count += 1

    df = pd.DataFrame(tweets_list)

    # Feature engineering (same as your original code)
    df["created_at"] = pd.to_datetime(df["created_at"])
    df["date"] = df["created_at"].dt.date
    df["hour"] = df["created_at"].dt.hour

    print(f"Successfully collected {len(df)} tweets from @{handle}")

    print(df.head())

RuntimeError: asyncio.run() cannot be called from a running event loop